# Project 1 — End-to-End ML Data Preprocessing Pipeline

Data Inspection 

Duplicate Removal 

Missing Value Handling 

Invalid Data Detection

Outlier Detection 

Feature Engineering 

Normalization 

Statistical Analysis 

Final Clean Dataset

 practice: vectorization, broadcasting, correlation, covariance, and distance calculations.


## Setup & Raw Dataset

In [2]:
import numpy as np

np.set_printoptions(suppress=True, precision=2)

# Columns: [Employee ID, Age, Height(cm), Weight(kg), Salary, Experience(years)]
employees = np.array([
    [101, 25, 170, 65, 50000, 3],
    [102, 30, 180, 75, 62000, 5],
    [103, np.nan, 175, 68, 58000, 4],
    [104, 22, 165, np.nan, 45000, 2],
    [105, 40, 172, 70, 1200000, 12],     # Salary Outlier
    [106, 35, 45, 80, 65000, 8],          # Invalid Height
    [107, -5, 169, 66, 52000, 2],         # Invalid Age
    [108, 28, 171, 67, -15000, 4],        # Invalid Salary
    [109, 32, 174, 69, 61000, np.nan],
    [102, 30, 180, 75, 62000, 5],         # Duplicate
    [110, 29, 168, 64, 57000, 3],
    [111, 31, 300, 72, 59000, 6],         # Invalid Height
    [112, 27, 173, 160, 56000, 4],        # Invalid Weight
    [113, 26, 176, 71, 54000, 3],
    [114, 33, 177, 73, 64000, 7]
], dtype=float)

COLS = ["EmployeeID", "Age", "Height_cm", "Weight_kg", "Salary", "Experience_yrs"]

print("Shape:", employees.shape)
employees


Shape: (15, 6)


array([[    101.,      25.,     170.,      65.,   50000.,       3.],
       [    102.,      30.,     180.,      75.,   62000.,       5.],
       [    103.,      nan,     175.,      68.,   58000.,       4.],
       [    104.,      22.,     165.,      nan,   45000.,       2.],
       [    105.,      40.,     172.,      70., 1200000.,      12.],
       [    106.,      35.,      45.,      80.,   65000.,       8.],
       [    107.,      -5.,     169.,      66.,   52000.,       2.],
       [    108.,      28.,     171.,      67.,  -15000.,       4.],
       [    109.,      32.,     174.,      69.,   61000.,      nan],
       [    102.,      30.,     180.,      75.,   62000.,       5.],
       [    110.,      29.,     168.,      64.,   57000.,       3.],
       [    111.,      31.,     300.,      72.,   59000.,       6.],
       [    112.,      27.,     173.,     160.,   56000.,       4.],
       [    113.,      26.,     176.,      71.,   54000.,       3.],
       [    114.,      33.,     17

## Phase 1 — Data Inspection

In [3]:
print("Rows, Columns:", employees.shape)
print("Dtype:", employees.dtype)
print()

# Missing values per column
nan_counts = np.isnan(employees).sum(axis=0)  
for name, cnt in zip(COLS, nan_counts):
    print(f"{name:15s} missing values: {int(cnt)}")


Rows, Columns: (15, 6)
Dtype: float64

EmployeeID      missing values: 0
Age             missing values: 1
Height_cm       missing values: 0
Weight_kg       missing values: 1
Salary          missing values: 0
Experience_yrs  missing values: 1


In [4]:
# Quick descriptive statistics (ignoring NaNs), column-wise using broadcasting-friendly axis ops
print(f"{'Column':15s}{'Min':>10s}{'Max':>10s}{'Mean':>10s}{'Std':>10s}")
for i, name in enumerate(COLS):
    col = employees[:, i]
    print(f"{name:15s}{np.nanmin(col):10.2f}{np.nanmax(col):10.2f}{np.nanmean(col):10.2f}{np.nanstd(col):10.2f}")


Column                Min       Max      Mean       Std
EmployeeID         101.00    114.00    107.13      4.13
Age                 -5.00     40.00     27.36      9.95
Height_cm           45.00    300.00    173.00     46.73
Weight_kg           64.00    160.00     76.79     23.47
Salary          -15000.001200000.00 128666.67 286938.24
Experience_yrs       2.00     12.00      4.86      2.61


##  Phase 2 — Duplicate Removal

Row `[102, 30, 180, 75, 62000, 5]` appears twice (Employee ID 102). We remove exact
duplicate rows, keeping the first occurrence. `np.unique` sorts by default, so we
instead find duplicate row indices manually to preserve original order.


In [5]:
def drop_duplicate_rows(data):
    seen = set()
    keep_idx = []
    for i, row in enumerate(data):
        # np.nan != np.nan, so we build a hashable key that treats NaN consistently
        key = tuple(np.round(row, 6))
        if key not in seen:
            seen.add(key)
            keep_idx.append(i)
    return data[keep_idx], keep_idx

employees_dedup, kept_idx = drop_duplicate_rows(employees)
print(f"Rows before: {employees.shape[0]}  |  Rows after: {employees_dedup.shape[0]}")
print("Removed row indices:", sorted(set(range(employees.shape[0])) - set(kept_idx)))
employees_dedup


Rows before: 15  |  Rows after: 14
Removed row indices: [9]


array([[    101.,      25.,     170.,      65.,   50000.,       3.],
       [    102.,      30.,     180.,      75.,   62000.,       5.],
       [    103.,      nan,     175.,      68.,   58000.,       4.],
       [    104.,      22.,     165.,      nan,   45000.,       2.],
       [    105.,      40.,     172.,      70., 1200000.,      12.],
       [    106.,      35.,      45.,      80.,   65000.,       8.],
       [    107.,      -5.,     169.,      66.,   52000.,       2.],
       [    108.,      28.,     171.,      67.,  -15000.,       4.],
       [    109.,      32.,     174.,      69.,   61000.,      nan],
       [    110.,      29.,     168.,      64.,   57000.,       3.],
       [    111.,      31.,     300.,      72.,   59000.,       6.],
       [    112.,      27.,     173.,     160.,   56000.,       4.],
       [    113.,      26.,     176.,      71.,   54000.,       3.],
       [    114.,      33.,     177.,      73.,   64000.,       7.]])

##  Phase 3 — Missing Value Handling

Missing values exist in **Age**, **Weight**, and **Experience**. We impute each with
its column **median** (robust to outliers, unlike mean) computed with `np.nanmedian`,
then broadcast the fill value into every NaN slot for that column.


In [6]:
employees_filled = employees_dedup.copy()

for i, name in enumerate(COLS):
    col = employees_filled[:, i]
    n_missing = np.isnan(col).sum()
    if n_missing > 0:
        median_val = np.nanmedian(col)
        col[np.isnan(col)] = median_val   # broadcasting a scalar into masked positions
        print(f"Filled {int(n_missing)} missing value(s) in '{name}' with median = {median_val:.2f}")

employees_filled


Filled 1 missing value(s) in 'Age' with median = 29.00
Filled 1 missing value(s) in 'Weight_kg' with median = 70.00
Filled 1 missing value(s) in 'Experience_yrs' with median = 4.00


array([[    101.,      25.,     170.,      65.,   50000.,       3.],
       [    102.,      30.,     180.,      75.,   62000.,       5.],
       [    103.,      29.,     175.,      68.,   58000.,       4.],
       [    104.,      22.,     165.,      70.,   45000.,       2.],
       [    105.,      40.,     172.,      70., 1200000.,      12.],
       [    106.,      35.,      45.,      80.,   65000.,       8.],
       [    107.,      -5.,     169.,      66.,   52000.,       2.],
       [    108.,      28.,     171.,      67.,  -15000.,       4.],
       [    109.,      32.,     174.,      69.,   61000.,       4.],
       [    110.,      29.,     168.,      64.,   57000.,       3.],
       [    111.,      31.,     300.,      72.,   59000.,       6.],
       [    112.,      27.,     173.,     160.,   56000.,       4.],
       [    113.,      26.,     176.,      71.,   54000.,       3.],
       [    114.,      33.,     177.,      73.,   64000.,       7.]])

##  Phase 4 — Invalid Data Detection

Business rules for a plausible employee record:

| Feature | Valid range |
|---|---|
| Age | 18 – 65 |
| Height | 100 – 250 cm |
| Weight | 30 – 200 kg |
| Salary | > 0 |

Rows violating any rule (negative age, height of 45 cm or 300 cm, negative salary,
weight of 160 kg) are flagged and removed.


In [7]:
age, height, weight, salary = (employees_filled[:, 1], employees_filled[:, 2],
                                 employees_filled[:, 3], employees_filled[:, 4])

valid_mask = (
    (age >= 18) & (age <= 65) &
    (height >= 100) & (height <= 250) &
    (weight >= 30) & (weight <= 200) &
    (salary > 0)
)

print("Invalid rows detected (by EmployeeID):", employees_filled[~valid_mask, 0])
employees_valid = employees_filled[valid_mask]
print(f"Rows before: {employees_filled.shape[0]}  |  Rows after: {employees_valid.shape[0]}")
employees_valid


Invalid rows detected (by EmployeeID): [106. 107. 108. 111.]
Rows before: 14  |  Rows after: 10


array([[    101.,      25.,     170.,      65.,   50000.,       3.],
       [    102.,      30.,     180.,      75.,   62000.,       5.],
       [    103.,      29.,     175.,      68.,   58000.,       4.],
       [    104.,      22.,     165.,      70.,   45000.,       2.],
       [    105.,      40.,     172.,      70., 1200000.,      12.],
       [    109.,      32.,     174.,      69.,   61000.,       4.],
       [    110.,      29.,     168.,      64.,   57000.,       3.],
       [    112.,      27.,     173.,     160.,   56000.,       4.],
       [    113.,      26.,     176.,      71.,   54000.,       3.],
       [    114.,      33.,     177.,      73.,   64000.,       7.]])

##  Phase 5 — Outlier Detection

Employee 105's salary (1,200,000) is a clear outlier. We use the **IQR method**
on Salary: anything beyond `Q1 - 1.5*IQR` or `Q3 + 1.5*IQR` is flagged.


In [8]:
sal = employees_valid[:, 4]
Q1, Q3 = np.percentile(sal, [25, 75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

print(f"Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}")
print(f"Valid salary range: [{lower:.2f}, {upper:.2f}]")

outlier_mask = (sal < lower) | (sal > upper)
print("Outlier EmployeeIDs:", employees_valid[outlier_mask, 0])

employees_clean = employees_valid[~outlier_mask]
print(f"Rows before: {employees_valid.shape[0]}  |  Rows after: {employees_clean.shape[0]}")
employees_clean


Q1=54500.00, Q3=61750.00, IQR=7250.00
Valid salary range: [43625.00, 72625.00]
Outlier EmployeeIDs: [105.]
Rows before: 10  |  Rows after: 9


array([[  101.,    25.,   170.,    65., 50000.,     3.],
       [  102.,    30.,   180.,    75., 62000.,     5.],
       [  103.,    29.,   175.,    68., 58000.,     4.],
       [  104.,    22.,   165.,    70., 45000.,     2.],
       [  109.,    32.,   174.,    69., 61000.,     4.],
       [  110.,    29.,   168.,    64., 57000.,     3.],
       [  112.,    27.,   173.,   160., 56000.,     4.],
       [  113.,    26.,   176.,    71., 54000.,     3.],
       [  114.,    33.,   177.,    73., 64000.,     7.]])

## Phase 6 — Feature Engineering

We derive new, more informative features:

- **BMI** = weight(kg) / (height(m))²
- **Salary per Year of Experience** = salary / (experience + 1)  *(avoids divide-by-zero)*
- **Age Group** (bucketed): Young (<30), Mid (30-40), Senior (>40)


In [9]:
height_m = employees_clean[:, 2] / 100
bmi = employees_clean[:, 3] / (height_m ** 2)

salary_per_exp = employees_clean[:, 4] / (employees_clean[:, 5] + 1)

age_col = employees_clean[:, 1]
age_group = np.where(age_col < 30, 0, np.where(age_col <= 40, 1, 2))  # 0=Young,1=Mid,2=Senior

engineered = np.column_stack([employees_clean, bmi, salary_per_exp, age_group])
FEATURE_COLS = COLS + ["BMI", "Salary_per_Exp_Year", "AgeGroup"]

print("New feature columns added:", FEATURE_COLS[-3:])
engineered


New feature columns added: ['BMI', 'Salary_per_Exp_Year', 'AgeGroup']


array([[  101.  ,    25.  ,   170.  ,    65.  , 50000.  ,     3.  ,
           22.49, 12500.  ,     0.  ],
       [  102.  ,    30.  ,   180.  ,    75.  , 62000.  ,     5.  ,
           23.15, 10333.33,     1.  ],
       [  103.  ,    29.  ,   175.  ,    68.  , 58000.  ,     4.  ,
           22.2 , 11600.  ,     0.  ],
       [  104.  ,    22.  ,   165.  ,    70.  , 45000.  ,     2.  ,
           25.71, 15000.  ,     0.  ],
       [  109.  ,    32.  ,   174.  ,    69.  , 61000.  ,     4.  ,
           22.79, 12200.  ,     1.  ],
       [  110.  ,    29.  ,   168.  ,    64.  , 57000.  ,     3.  ,
           22.68, 14250.  ,     0.  ],
       [  112.  ,    27.  ,   173.  ,   160.  , 56000.  ,     4.  ,
           53.46, 11200.  ,     0.  ],
       [  113.  ,    26.  ,   176.  ,    71.  , 54000.  ,     3.  ,
           22.92, 13500.  ,     0.  ],
       [  114.  ,    33.  ,   177.  ,    73.  , 64000.  ,     7.  ,
           23.3 ,  8000.  ,     1.  ]])

##  Phase 7 — Normalization

We scale numeric features for ML readiness:

- **Min-Max scaling** → [0, 1] range, useful for distance-based models
- **Standardization (Z-score)** → mean 0, std 1, useful for linear models

We apply this to Age, Height, Weight, Salary, Experience, BMI, Salary_per_Exp_Year
(Employee ID and AgeGroup are identifiers/categorical, so we leave them out).


In [10]:
scale_idx = [1, 2, 3, 4, 5, 6, 7]  # Age, Height, Weight, Salary, Experience, BMI, Salary_per_Exp
X = engineered[:, scale_idx]

# Min-Max scaling (broadcasting min/max row-vectors across all rows)
X_min, X_max = X.min(axis=0), X.max(axis=0)
X_minmax = (X - X_min) / (X_max - X_min)

# Standardization (broadcasting mean/std row-vectors across all rows)
X_mean, X_std = X.mean(axis=0), X.std(axis=0)
X_standard = (X - X_mean) / X_std

print("Min-Max scaled features:")
print(X_minmax)
print()
print("Standardized (Z-score) features:")
print(X_standard)


Min-Max scaled features:
[[0.27 0.33 0.01 0.26 0.2  0.01 0.64]
 [0.73 1.   0.11 0.89 0.6  0.03 0.33]
 [0.64 0.67 0.04 0.68 0.4  0.   0.51]
 [0.   0.   0.06 0.   0.   0.11 1.  ]
 [0.91 0.6  0.05 0.84 0.4  0.02 0.6 ]
 [0.64 0.2  0.   0.63 0.2  0.02 0.89]
 [0.45 0.53 1.   0.58 0.4  1.   0.46]
 [0.36 0.73 0.07 0.47 0.2  0.02 0.79]
 [1.   0.8  0.09 1.   1.   0.04 0.  ]]

Standardized (Z-score) features:
[[-0.95 -0.7  -0.5  -1.12 -0.65 -0.42  0.22]
 [ 0.58  1.55 -0.16  1.    0.81 -0.35 -0.86]
 [ 0.27  0.43 -0.4   0.29  0.08 -0.45 -0.23]
 [-1.86 -1.83 -0.33 -2.   -1.38 -0.08  1.46]
 [ 1.19  0.2  -0.36  0.82  0.08 -0.39  0.07]
 [ 0.27 -1.15 -0.54  0.12 -0.65 -0.4   1.09]
 [-0.34 -0.03  2.81 -0.06  0.08  2.81 -0.43]
 [-0.64  0.65 -0.29 -0.41 -0.65 -0.38  0.72]
 [ 1.49  0.88 -0.22  1.35  2.27 -0.34 -2.03]]


##  Phase 8 — Statistical Analysis

We examine relationships between features:

- **Correlation matrix** — strength/direction of linear relationships (scale-independent)
- **Covariance matrix** — joint variability (scale-dependent)
- **Pairwise Euclidean distances** between employees (on standardized features), using
  vectorized broadcasting instead of nested loops


In [11]:
feat_names = [FEATURE_COLS[i] for i in scale_idx]

corr = np.corrcoef(X_standard, rowvar=False)
cov = np.cov(X_standard, rowvar=False)

print("Feature order:", feat_names)
print("\nCorrelation matrix:")
print(corr)
print("\nCovariance matrix:")
print(cov)


Feature order: ['Age', 'Height_cm', 'Weight_kg', 'Salary', 'Experience_yrs', 'BMI', 'Salary_per_Exp_Year']

Correlation matrix:
[[ 1.    0.66 -0.09  0.97  0.82 -0.17 -0.72]
 [ 0.66  1.    0.07  0.78  0.73 -0.06 -0.76]
 [-0.09  0.07  1.    0.03  0.09  0.99 -0.22]
 [ 0.97  0.78  0.03  1.    0.85 -0.08 -0.77]
 [ 0.82  0.73  0.09  0.85  1.    0.   -0.97]
 [-0.17 -0.06  0.99 -0.08  0.    1.   -0.12]
 [-0.72 -0.76 -0.22 -0.77 -0.97 -0.12  1.  ]]

Covariance matrix:
[[ 1.12  0.74 -0.1   1.09  0.92 -0.19 -0.81]
 [ 0.74  1.13  0.08  0.88  0.83 -0.06 -0.85]
 [-0.1   0.08  1.13  0.03  0.11  1.12 -0.24]
 [ 1.09  0.88  0.03  1.12  0.95 -0.08 -0.87]
 [ 0.92  0.83  0.11  0.95  1.12  0.   -1.09]
 [-0.19 -0.06  1.12 -0.08  0.    1.12 -0.13]
 [-0.81 -0.85 -0.24 -0.87 -1.09 -0.13  1.12]]


In [12]:
# Vectorized pairwise Euclidean distance matrix using broadcasting:
# ||a - b||^2 = ||a||^2 + ||b||^2 - 2*a.b
sq_norms = np.sum(X_standard**2, axis=1)
dist_sq = sq_norms[:, None] + sq_norms[None, :] - 2 * X_standard @ X_standard.T
dist_sq = np.maximum(dist_sq, 0)  # guard against tiny negative floating point noise
dist_matrix = np.sqrt(dist_sq)

emp_ids = engineered[:, 0].astype(int)
print("Employee IDs (row/col order):", emp_ids)
print("\nPairwise Euclidean distance matrix (standardized features):")
print(dist_matrix)

# Most similar pair (excluding self-distance of 0 on the diagonal)
np.fill_diagonal(dist_matrix, np.inf)
i, j = np.unravel_index(np.argmin(dist_matrix), dist_matrix.shape)
print(f"\nMost similar employees: ID {emp_ids[i]} and ID {emp_ids[j]} "
      f"(distance = {dist_matrix[i, j]:.3f})")


Employee IDs (row/col order): [101 102 103 104 109 110 112 113 114]

Pairwise Euclidean distance matrix (standardized features):
[[0.   3.91 2.34 2.26 3.11 1.99 4.93 1.65 5.31]
 [3.91 0.   1.69 6.06 1.92 3.78 4.9  2.99 2.22]
 [2.34 1.69 0.   4.48 1.12 2.2  4.66 1.68 3.3 ]
 [2.26 6.06 4.48 0.   5.05 3.21 5.77 3.37 7.44]
 [3.11 1.92 1.12 5.05 0.   2.18 4.87 2.46 3.17]
 [1.99 3.78 2.2  3.21 2.18 0.   5.11 2.14 5.05]
 [4.93 4.9  4.66 5.77 4.87 5.11 0.   4.73 5.71]
 [1.65 2.99 1.68 3.37 2.46 2.14 4.73 0.   4.88]
 [5.31 2.22 3.3  7.44 3.17 5.05 5.71 4.88 0.  ]]

Most similar employees: ID 103 and ID 109 (distance = 1.123)


##  Phase 9 — Final Clean Dataset

The pipeline output: a clean, engineered, and scaled dataset — ready for a machine
learning model. We assemble a final table combining IDs, engineered raw features, and
the standardized ML-ready feature matrix.


In [14]:
print("=== PIPELINE SUMMARY ===")
print(f"Original rows:            {employees.shape[0]}")
print(f"After duplicate removal:  {employees_dedup.shape[0]}")
print(f"After missing-value fill: {employees_filled.shape[0]}  (no rows dropped, only imputed)")
print(f"After invalid-data drop:  {employees_valid.shape[0]}")
print(f"After outlier removal:    {employees_clean.shape[0]}  <-- final row count")
print(f"Final feature count:      {len(feat_names)} numeric features scaled + AgeGroup category")
print()

print("Final engineered (unscaled) dataset:")
print(FEATURE_COLS)
print(engineered)

print("\nFinal ML-ready standardized feature matrix (X_standard), row-aligned to EmployeeID:")
final_table = np.column_stack([emp_ids, X_standard])
print(["EmployeeID"] + feat_names)
print(final_table)


=== PIPELINE SUMMARY ===
Original rows:            15
After duplicate removal:  14
After missing-value fill: 14  (no rows dropped, only imputed)
After invalid-data drop:  10
After outlier removal:    9  <-- final row count
Final feature count:      7 numeric features scaled + AgeGroup category

Final engineered (unscaled) dataset:
['EmployeeID', 'Age', 'Height_cm', 'Weight_kg', 'Salary', 'Experience_yrs', 'BMI', 'Salary_per_Exp_Year', 'AgeGroup']
[[  101.      25.     170.      65.   50000.       3.      22.49 12500.
      0.  ]
 [  102.      30.     180.      75.   62000.       5.      23.15 10333.33
      1.  ]
 [  103.      29.     175.      68.   58000.       4.      22.2  11600.
      0.  ]
 [  104.      22.     165.      70.   45000.       2.      25.71 15000.
      0.  ]
 [  109.      32.     174.      69.   61000.       4.      22.79 12200.
      1.  ]
 [  110.      29.     168.      64.   57000.       3.      22.68 14250.
      0.  ]
 [  112.      27.     173.     160.   56000